# 对接 TensorFlow 项目实践

上一节介绍了 AutoFuse 对接 PyTorch 项目的基本方法，本节转向 TensorFlow 1.x 图模式，以 `Abs → ReLU → Exp` 逐元素计算图为例（输入为 `[128, 192]` 的 `float16` 张量，开启 AutoFuse 后，三个算子可构成候选融合范围参与图编译），完成环境配置、NPU 图执行、数值校验、融合结果检查、性能分析和常见问题排查，并确认数值正确性与融合结果。

本节学习大纲如下：

- 设置运行环境
- 执行 TensorFlow 自动融合用例
- 验证运行与融合结果
- TensorFlow 性能分析
- TensorFlow 常见问题

## 1. 设置运行环境

TensorFlow GE 路线通过环境变量 `AUTOFUSE_FLAGS` 启用 AutoFuse，该变量必须在导入 TensorFlow 和 `npu_bridge` 前生效：

```bash
export AUTOFUSE_FLAGS="--enable_autofuse=true"
```

本节用例以子进程脚本方式执行，执行单元中会先导出该变量再启动脚本，确保变量在框架初始化前生效。修改变量取值后，应重新执行完整用例。

执行用例前，还需完成 CANN 环境初始化（默认已初始化），并确保当前 Python 环境已安装与 TensorFlow 1.x 匹配的 NPU 插件（`npu_bridge`）。设置成功不等于目标算子一定会融合，最终仍需通过 Profiling 或调试产物确认。

> **注意**：如果已在 Notebook 内核中导入 TensorFlow 或 `npu_bridge`，子进程方式不受影响，可直接重新执行用例单元。

## 2. 执行 TensorFlow 自动融合用例（目前环境暂不支持执行）

下面的单元先将完整用例写入 `./tmp/autofuse_tensorflow_example.py`，再导出环境变量并用 Python 在子进程中执行。子进程方式保证环境变量在框架初始化前生效，每次运行都是全新进程，也不受内核中残留模块状态影响。

脚本按以下顺序执行：

1. 导入 `numpy`、`tensorflow` 和 `npu_bridge`，其中导入 `npu_bridge` 是在框架初始化阶段注册 NPU 能力的关键。
2. 将 `NpuOptimizer` 添加到 TensorFlow 会话配置，并打开 Profiling。
3. 构建 `Abs → ReLU → Exp` 计算图，以固定随机输入执行 100 步。
4. 使用相同输入计算 NumPy 参考结果并执行精度断言。
5. 导出本次运行新生成的 Profiling 数据，并打印结果形状、数据类型、最大绝对误差和输出目录。

完整用例集中在一个脚本中，便于保持图定义、会话执行和结果验证的一致性。执行单元会先切换到 `tmp` 目录再启动脚本，使脚本内相对路径以 `tmp` 为基准。CANN 环境由 Notebook 运行环境自动加载，无需在单元内手动加载。

In [ ]:
%%writefile ./tmp/autofuse_tensorflow_example.py
import glob
import os
import subprocess

import numpy as np
import tensorflow as tf
import npu_bridge

_ = npu_bridge  # 通过 import 副作用注册 NPU 算子

PROFILING_DIR = os.path.abspath("./profiling")
PROFILING_OPTIONS = (
    '{"output":"%s","training_trace":"on","task_time":"on",'
    '"hccl":"on","aicpu":"on","aic_metrics":"PipeUtilization","msproftx":"off"}'
) % PROFILING_DIR


def configure_npu(session_config):
    """在 TensorFlow 会话配置中添加 NpuOptimizer 并开启 Profiling。"""
    custom_op = session_config.graph_options.rewrite_options.custom_optimizers.add()
    custom_op.name = "NpuOptimizer"
    custom_op.parameter_map["use_off_line"].b = True
    custom_op.parameter_map["graph_run_mode"].i = 0
    custom_op.parameter_map["profiling_mode"].b = True
    custom_op.parameter_map["profiling_options"].s = tf.compat.as_bytes(PROFILING_OPTIONS)
    return session_config


def get_profile_dirs():
    """返回当前 Profiling 输出目录集合。"""
    return set(glob.glob(os.path.join(PROFILING_DIR, "PROF_*")))


def export_new_profiling(profile_dirs_before):
    """导出本次执行新生成的 Profiling 数据。"""
    new_profile_dirs = sorted(get_profile_dirs() - profile_dirs_before)
    for profile_dir in new_profile_dirs:
        subprocess.run(
            ["msprof", "--export=on", "--output={}".format(profile_dir)],
            check=True,
        )
    return new_profile_dirs


def run_model(step=100):
    """构建计算图，在 NPU 上执行并返回结果、误差和 Profiling 目录。"""
    profile_dirs_before = get_profile_dirs()
    input_tensor = tf.placeholder(tf.float16, shape=[128, 192], name="input")
    abs_result = tf.abs(input_tensor, name="abs")
    relu_result = tf.nn.relu(abs_result, name="relu")
    output_tensor = tf.exp(relu_result, name="exp")

    np.random.seed(0)
    input_data = np.random.uniform(-1.0, 1.0, size=(128, 192)).astype(np.float16)
    expected = np.exp(np.maximum(np.abs(input_data), 0)).astype(np.float16)

    session_config = tf.ConfigProto(
        allow_soft_placement=True,
        log_device_placement=False,
    )
    configure_npu(session_config)

    with tf.Session(config=session_config) as session:
        for _ in range(step):
            actual = session.run(output_tensor, feed_dict={input_tensor: input_data})

    max_abs_error = float(np.max(np.abs(actual.astype(np.float32) - expected.astype(np.float32))))
    np.testing.assert_allclose(actual, expected, rtol=1e-3, atol=1e-3)
    profile_dirs = export_new_profiling(profile_dirs_before)
    return actual, max_abs_error, profile_dirs


actual, max_abs_error, profile_dirs = run_model()
print("TensorFlow version:", tf.__version__)
print("Output shape:", actual.shape)
print("Output dtype:", actual.dtype)
print("Max absolute error:", max_abs_error)
print("New profiling directories:", profile_dirs)

In [ ]:
%%bash
set -eo pipefail

# 环境变量需在 TensorFlow 和 npu_bridge 导入前生效，此处统一导出
export AUTOFUSE_FLAGS="--enable_autofuse=true"

# 切换到 tmp 目录执行，使脚本内相对路径以 tmp 为基准
cd ./tmp

# CANN 环境已由 Notebook 运行环境自动加载，直接执行用例脚本
python3 ./autofuse_tensorflow_example.py

## 3. 验证运行与融合结果

代码执行完成后，需要分别验证“计算是否正确”和“算子是否融合”。二者关注点不同，不能只以代码无报错作为融合成功的依据。

### 3.1 理解运行输出

脚本执行完成后会打印以下信息：

<table align="left">
<thead><tr><th>输出项</th><th>预期观察</th><th>说明</th></tr></thead>
<tbody>
<tr><td><code>TensorFlow version</code></td><td>当前 TensorFlow 1.x 版本</td><td>用于确认执行环境是否符合本例要求</td></tr>
<tr><td><code>Output shape</code></td><td><code>(128, 192)</code></td><td>输出形状应与输入形状一致</td></tr>
<tr><td><code>Output dtype</code></td><td><code>float16</code></td><td>输出数据类型应与输入一致</td></tr>
<tr><td><code>Max absolute error</code></td><td>满足代码中的误差阈值</td><td>表示 NPU 输出与 NumPy 参考结果接近；超出阈值时断言会报错</td></tr>
<tr><td><code>New profiling directories</code></td><td>包含本次新生成的目录</td><td>后续从这些目录中检查算子执行与融合信息</td></tr>
</tbody>
</table>
<div style="clear:left"></div>

### 3.2 验证数值结果

代码使用 `np.testing.assert_allclose` 对比 NPU 输出和 NumPy 参考结果。断言通过说明结果位于当前示例设置的相对误差与绝对误差范围内；断言失败时，应先排查输入、数据类型、参考计算和首个产生差异的算子，再分析融合行为。

### 3.3 验证融合结果

运行完成后，`tmp` 目录下会生成 `profiling` 目录及相关 Profiling 文件。TensorFlow Profiling 导出结果通常位于：

```text
./tmp/profiling/
└── PROF_000001_时间戳_随机串/
    └── mindstudio_profiler_output/
        └── op_summary_*.csv
```

打开本次运行对应的 `op_summary_*.csv`，搜索 `Abs`、`Relu`、`Exp` 或 `autofuse`。预期可观察到类似 `autofuse_pointwise_0_Abs_Relu_Exp` 的融合 Kernel，表明三个逐元素算子进入同一融合范围。具体名称可能随版本变化，因此应结合算子数量、算子类型和执行记录综合判断，避免仅依赖固定名称。

### 3.4 算子自动融合执行结果

<div style="text-align:left">
<img src="./images/tensorflow_autofuse.png" alt="TensorFlow 路线自动融合执行结果" width="60%">
</div>

该图展示开启 AutoFuse 后的 Profiling 结果。`Abs`、`ReLU` 和 `Exp` 由一个融合 Kernel 执行，说明目标算子链已进入同一融合范围。

### 3.5 单算子执行结果

如需获取单算子场景的对照结果，可将执行单元中的 `export AUTOFUSE_FLAGS="--enable_autofuse=true"` 一行注释掉（或改为 `--enable_autofuse=false`），再重新运行完整用例，并使用相同的输入、迭代次数和 Profiling 配置。

<div style="text-align:left">
<img src="./images/tensorflow_no_autofuse.png" alt="TensorFlow 路线单算子执行结果" width="80%">
</div>

该图展示未开启 AutoFuse 时的 Profiling 结果。`Abs`、`ReLU` 和 `Exp` 分别以独立算子的形式执行，可作为融合场景的对照。

对比两种场景时，应比较融合 Kernel 的平均任务执行时间与单算子场景下三个独立算子的平均任务执行时间之和。若融合 Kernel 的平均执行时间更小，说明本次实验中融合减少了整体执行开销并取得了性能收益。首次图编译开销应与预热后的稳态执行时间分开统计。

## 4. TensorFlow 性能分析

本例的性能分析需要区分首次图编译开销和预热后的稳态执行耗时。首次执行会包含图准备和编译过程，耗时明显偏长，不应直接作为稳态性能结果。

建议按以下方式进行对比：

1. 准备融合与基线两种场景：分别开启和关闭 AutoFuse，并重新执行完整用例（参见 3.5 节）。
2. 两种场景使用相同的输入数据、数据类型、执行步数、预热方式和 Profiling 配置，保证结果可比。
3. 完成预热后再采样：排除首次图编译开销，比较多轮迭代的平均执行时间。
4. 结合 Profiling 中的 Kernel 数量、融合 Kernel 名称和平均任务执行时间进行判断。

出现 `autofuse_` 开头的融合 Kernel 只能说明算子发生了融合。只有当融合 Kernel 的稳态执行时间低于基线场景中 `Abs`、`ReLU` 和 `Exp` 等独立算子的执行时间之和时，才能说明本次实验取得了性能收益。小规模算子链的融合收益可能不明显。

## 5. TensorFlow 常见问题

- **环境变量未生效**：确认执行单元中 `AUTOFUSE_FLAGS` 的导出位于用例脚本启动之前；修改变量后，应重新执行完整用例。
- **`npu_bridge` 无法导入**：确认 TensorFlow、TF Adapter、CANN 和 Python 版本匹配，并检查 CANN 环境是否已初始化。
- **未生成融合 Kernel**：确认 `NpuOptimizer` 配置正确，AutoFuse 已开启，输入计算图仍包含目标算子链，并结合 Profiling 检查实际执行记录。
- **未生成 Profiling**：确认 Profiling 配置在编译前生效，检查输出目录权限，并区分本次产物与历史产物。
- **精度异常**：使用相同输入对比 NPU 输出和 NumPy 参考结果，确认数据类型、参考计算和误差阈值一致，再定位首个产生差异的算子。


## 课后练习

请根据本节实践内容完成以下题目进行自测。

1. （判断题）TensorFlow GE 路线中，`AUTOFUSE_FLAGS` 需要在导入 TensorFlow 和 `npu_bridge` 之前设置才能生效。

2. （判断题）TensorFlow 用例执行无报错即可证明目标算子（`Abs`、`ReLU`、`Exp`）已经完成自动融合，无需额外检查 Profiling 数据。

3. （多选题）未观察到融合 Kernel 时，应检查哪些方向？
    A. `AUTOFUSE_FLAGS` 是否在框架初始化前生效
    B. Profiling 采集与导出是否成功
    C. 实际计算图是否仍包含目标算子链
    D. Markdown 文档展示样式

4. （多选题）本节通过哪些方式验证 TensorFlow 用例的运行结果？
    A. NumPy 参考结果对比
    B. 结果形状与最大绝对误差
    C. Profiling 数据检查融合 Kernel
    D. 仅观察 Python 进程是否正常退出

**执行以下代码获取答案。**

In [ ]:
!cat ./answer/03.04_answer.txt